In [1]:
%pip install langchain-core "langchain-community==0.3.31" langchain-groq pymysql cryptography python-dotenv ragas pandas streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import warnings
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore", category=DeprecationWarning)  # hides the harmless RAGAS deprecation notices
load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file"
assert os.getenv("DB_PASSWORD"), "Add DB_PASSWORD to your .env file"

In [3]:
host = "localhost"
port = "3306"
username = "root"
password = quote_plus(os.getenv("DB_PASSWORD"))
database_schema = "text_to_sql"

mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"
engine = create_engine(mysql_uri)
db = SQLDatabase(engine, sample_rows_in_table_info=2)

print(db.get_usable_table_names())

['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'state_regions']


In [4]:
context = db.get_table_info()
print(context[:1500])  # preview


CREATE TABLE `2017_budgets` (
	`Product Name` TEXT, 
	`2017 Budgets` DOUBLE
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
2 rows from 2017_budgets table:
Product Name	2017 Budgets
Product 1	3016489.2089999998
Product 2	3050087.5649999999
*/


CREATE TABLE customers (
	`Customer Index` INTEGER, 
	`Customer Names` TEXT
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
2 rows from customers table:
Customer Index	Customer Names
1	Geiss Company
2	Jaxbean Group
*/


CREATE TABLE products (
	`Index` INTEGER, 
	`Product Name` TEXT
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
2 rows from products table:
Index	Product Name
1	Product 1
2	Product 2
*/


CREATE TABLE regions (
	id INTEGER, 
	name TEXT, 
	county TEXT, 
	state_code TEXT, 
	state TEXT, 
	type TEXT, 
	latitude DOUBLE, 
	longitude DOUBLE, 
	area_code INTEGER, 
	population INTEGER, 
	households INTEGER, 
	median_income INTEGER, 
	land_area INTEGER, 
	water_area INTEGER

In [5]:
template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide me the sql query dont include anything else.
           Provide me sql query in a single line dont add line breaks.
           Use backticks around table and column names that contain spaces.
Table Schema:
{schema}

Question: {question}
SQL Query:
"""
prompt = ChatPromptTemplate.from_template(template)

In [6]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [7]:
sql_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [8]:
# Create the SQL query chain using the LLM and the prompt template
sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: get_schema(db))
    | prompt
    | sql_llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [9]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question": "What was the budget of Product 12"})
print(resp)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';


In [10]:
#test the SQL query chain with a sample question
resp1=sql_chain.invoke({"question": "What was the total linetotal for the geiss company"})
print(resp1)

SELECT SUM(`Line Total`) AS total_linetotal FROM sales_order AS so JOIN customers AS c ON so.`Customer Name Index` = c.`Customer Index` WHERE c.`Customer Names` = 'Geiss Company';


In [11]:
import re

match = re.search(r"```(?:sql)?\s*(.*?)\s*```", resp, re.DOTALL | re.IGNORECASE)
query = match.group(1).strip() if match else resp.strip()



In [12]:
query

"SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';"

In [13]:
db.run(query)

'[(1356976.996,)]'

### Evaluation

Execution Accuracy

RAGAS Context Precision

In [14]:
from ragas import evaluate
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import ContextPrecision

evaluator_llm = LangchainLLMWrapper(sql_llm)
context_precision = ContextPrecision(llm=evaluator_llm)

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
user_inputs = [
    # simple lookups
    "What was the budget of Product 12",
    "What are the names of all products in the products table?",
    "List all customer names from the customers table.",
    "Find the name and state of all regions in the regions table.",
    "What is the name of the customer with Customer Index = 1",
    # harder: aggregation, group by, order by + limit, joins
    "What is the total line total for each sales channel? Show the channel and the total.",
    "Who are the top 5 customers by total line total? Show the customer name and the total.",
    "Which product has the highest total order quantity? Show the product name and the total quantity.",
    "Which 3 states have the highest total population in the regions table? Show the state and the total population.",
    "What is the total line total of orders delivered to regions in the state of California?",
    "How many customers are there?",
]

references = [
    "SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';",
    "SELECT `Product Name` FROM products;",
    "SELECT `Customer Names` FROM customers;",
    "SELECT name, state FROM regions;",
    "SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1;",
    "SELECT Channel, SUM(`Line Total`) FROM sales_order GROUP BY Channel;",
    "SELECT c.`Customer Names`, SUM(s.`Line Total`) FROM sales_order s "
    "JOIN customers c ON s.`Customer Name Index` = c.`Customer Index` "
    "GROUP BY c.`Customer Names` ORDER BY SUM(s.`Line Total`) DESC LIMIT 5;",
    "SELECT p.`Product Name`, SUM(s.`Order Quantity`) FROM sales_order s "
    "JOIN products p ON s.`Product Description Index` = p.`Index` "
    "GROUP BY p.`Product Name` ORDER BY SUM(s.`Order Quantity`) DESC LIMIT 1;",
    "SELECT state, SUM(population) FROM regions GROUP BY state ORDER BY SUM(population) DESC LIMIT 3;",
    "SELECT SUM(s.`Line Total`) FROM sales_order s "
    "JOIN regions r ON s.`Delivery Region Index` = r.id WHERE r.state = 'California';",
    "SELECT COUNT(*) FROM customers;",
]

assert len(user_inputs) == len(references)

In [16]:
def run_df(query: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query.rstrip(";")), conn)


def norm_val(v):
    """Round numbers so tiny floating-point differences don't count as a mismatch."""
    try:
        return f"{float(v):.4f}"
    except (TypeError, ValueError):
        return str(v)


def normalize(df: pd.DataFrame):
    """Compare values only: ignores column names/aliases and row order."""
    return sorted(tuple(norm_val(v) for v in row) for row in df.values.tolist())

checking whether every reference we gave ran

In [17]:
for q in references:
    try:
        n = len(run_df(q))
        print(f"[OK]   {n} row(s) | {q[:90]}")
    except Exception as e:
        print(f"[FAIL] {q[:90]}\n       {str(e).splitlines()[0][:200]}")

[OK]   1 row(s) | SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';
[OK]   30 row(s) | SELECT `Product Name` FROM products;
[OK]   175 row(s) | SELECT `Customer Names` FROM customers;
[OK]   994 row(s) | SELECT name, state FROM regions;
[OK]   1 row(s) | SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1;
[OK]   3 row(s) | SELECT Channel, SUM(`Line Total`) FROM sales_order GROUP BY Channel;
[OK]   5 row(s) | SELECT c.`Customer Names`, SUM(s.`Line Total`) FROM sales_order s JOIN customers c ON s.`C
[OK]   1 row(s) | SELECT p.`Product Name`, SUM(s.`Order Quantity`) FROM sales_order s JOIN products p ON s.`
[OK]   3 row(s) | SELECT state, SUM(population) FROM regions GROUP BY state ORDER BY SUM(population) DESC LI
[OK]   1 row(s) | SELECT SUM(s.`Line Total`) FROM sales_order s JOIN regions r ON s.`Delivery Region Index` 
[OK]   1 row(s) | SELECT COUNT(*) FROM customers;


Generating SQL for every test question

In [18]:
responses = []

for question in user_inputs:
    resp = sql_chain.invoke({"question": question})
    match = re.search(r"```(?:sql)?\s*(.*?)\s*```", resp, re.DOTALL | re.IGNORECASE)
    responses.append(match.group(1).strip() if match else resp.strip())

for q, r in zip(user_inputs, responses):
    print(q, "\n  ->", r, "\n")

What was the budget of Product 12 
  -> SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'; 

What are the names of all products in the products table? 
  -> SELECT `Product Name` FROM products; 

List all customer names from the customers table. 
  -> SELECT `Customer Names` FROM customers; 

Find the name and state of all regions in the regions table. 
  -> SELECT name, state FROM regions; 

What is the name of the customer with Customer Index = 1 
  -> SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1; 

What is the total line total for each sales channel? Show the channel and the total. 
  -> SELECT `Channel`, SUM(`Line Total`) AS total FROM sales_order GROUP BY `Channel`; 

Who are the top 5 customers by total line total? Show the customer name and the total. 
  -> SELECT c.`Customer Names` AS customer_name, SUM(so.`Line Total`) AS total FROM sales_order so JOIN customers c ON so.`Customer Name Index` = c.`Customer Index` GROUP BY c.`Cust

Execution Accuarcy

In [19]:
exec_correct = []

for question, gen, ref in zip(user_inputs, responses, references):
    try:
        ok = normalize(run_df(gen)) == normalize(run_df(ref))
    except Exception as e:
        ok = False
        print(f"ERROR running query for: {question}\n   {str(e).splitlines()[0][:150]}")
    exec_correct.append(ok)

for question, gen, ok in zip(user_inputs, responses, exec_correct):
    print("PASS" if ok else "FAIL", "|", question)
    if not ok:
        print("       generated:", gen)

print(f"\nExecution accuracy: {sum(exec_correct)}/{len(exec_correct)} = {sum(exec_correct)/len(exec_correct):.0%}")

PASS | What was the budget of Product 12
PASS | What are the names of all products in the products table?
PASS | List all customer names from the customers table.
PASS | Find the name and state of all regions in the regions table.
PASS | What is the name of the customer with Customer Index = 1
PASS | What is the total line total for each sales channel? Show the channel and the total.
PASS | Who are the top 5 customers by total line total? Show the customer name and the total.
PASS | Which product has the highest total order quantity? Show the product name and the total quantity.
PASS | Which 3 states have the highest total population in the regions table? Show the state and the total population.
PASS | What is the total line total of orders delivered to regions in the state of California?
PASS | How many customers are there?

Execution accuracy: 11/11 = 100%


RAGAS Evaluation

In [20]:
retrieved_contexts = [context]

samples = [
    SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    for i in range(len(user_inputs))
]

ragas_eval_dataset = EvaluationDataset(samples=samples)
ragas_eval_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,What was the budget of Product 12,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...
1,What are the names of all products in the prod...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Product Name` FROM products;,SELECT `Product Name` FROM products;
2,List all customer names from the customers table.,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers;,SELECT `Customer Names` FROM customers;
3,Find the name and state of all regions in the ...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT name, state FROM regions;","SELECT name, state FROM regions;"
4,What is the name of the customer with Customer...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers WHERE `...,SELECT `Customer Names` FROM customers WHERE `...
5,What is the total line total for each sales ch...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT `Channel`, SUM(`Line Total`) AS total F...","SELECT Channel, SUM(`Line Total`) FROM sales_o..."
6,Who are the top 5 customers by total line tota...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT c.`Customer Names` AS customer_name, SU...","SELECT c.`Customer Names`, SUM(s.`Line Total`)..."
7,Which product has the highest total order quan...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT p.`Product Name`, SUM(s.`Order Quantity...","SELECT p.`Product Name`, SUM(s.`Order Quantity..."
8,Which 3 states have the highest total populati...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT state, SUM(population) AS total_populat...","SELECT state, SUM(population) FROM regions GRO..."
9,What is the total line total of orders deliver...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT SUM(`Line Total`) AS total_line_total F...,SELECT SUM(s.`Line Total`) FROM sales_order s ...


In [21]:
result = evaluate(
    dataset=ragas_eval_dataset,
    metrics=[context_precision],
)
result

Evaluating: 100%|██████████| 11/11 [02:37<00:00, 14.33s/it]


{'context_precision': 1.0000}

Final Results Table

In [22]:
df = result.to_pandas()
df["execution_correct"] = exec_correct
df

,user_input,retrieved_contexts,response,reference,context_precision,execution_correct
0,What was the budget of Product 12,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...,1.0,True
1,What are the names of all products in the prod...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Product Name` FROM products;,SELECT `Product Name` FROM products;,1.0,True
2,List all customer names from the customers table.,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers;,SELECT `Customer Names` FROM customers;,1.0,True
3,Find the name and state of all regions in the ...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT name, state FROM regions;","SELECT name, state FROM regions;",1.0,True
4,What is the name of the customer with Customer...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers WHERE `...,SELECT `Customer Names` FROM customers WHERE `...,1.0,True
5,What is the total line total for each sales ch...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT `Channel`, SUM(`Line Total`) AS total F...","SELECT Channel, SUM(`Line Total`) FROM sales_o...",1.0,True
6,Who are the top 5 customers by total line tota...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT c.`Customer Names` AS customer_name, SU...","SELECT c.`Customer Names`, SUM(s.`Line Total`)...",1.0,True
7,Which product has the highest total order quan...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT p.`Product Name`, SUM(s.`Order Quantity...","SELECT p.`Product Name`, SUM(s.`Order Quantity...",1.0,True
8,Which 3 states have the highest total populati...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT state, SUM(population) AS total_populat...","SELECT state, SUM(population) FROM regions GRO...",1.0,True
9,What is the total line total of orders deliver...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT SUM(`Line Total`) AS total_line_total F...,SELECT SUM(s.`Line Total`) FROM sales_order s ...,1.0,True


In [23]:
# The Streamlit app's "Evaluation" page reads this file
df.drop(columns=["retrieved_contexts"]).to_csv("eval_results.csv", index=False)
print("Saved eval_results.csv")

Saved eval_results.csv
